# ExaRanker (2023)
---
[[paper]](https://arxiv.org/pdf/2305.15609)
ExaRanker = eXtreme-scale Ranker
Модель предложена исследователями из Google Research в 2023 году.

ExaRanker — это высокоэффективная и точная модель для **нейронного реранкинга (neural reranking)**, разработанная для обработки экстремально больших списков документов в задачах информационного поиска, преодолевающая ограничения традиционных кросс-энкодеров на больших масштабах.

### Контекст
В современных системах информационного поиска и RAG (Retrieval-Augmented Generation) пайплайнах ключевую роль играет многоступенчатый подход: сначала быстрый, но менее точный ретривер (например, BM25, DPR (2020), ColBERT (2020)) извлекает широкий набор потенциально релевантных документов, а затем более сложная модель-реранкер уточняет их порядок, чтобы выбрать наиболее релевантные. Однако традиционные мощные реранкеры, основанные на кросс-энкодерах, обладают высокой вычислительной сложностью (квадратичной по длине документа и линейной по количеству документов), что делает их медленными и дорогостоящими для обработки списков из сотен или тысяч документов. Это ограничение часто называют "проклятием N-ранга" (curse of N-rank), вынуждая использовать реранкеры только для очень коротких списков документов (например, топ-100).

### Идея
Основная идея ExaRanker состоит в том, чтобы сделать нейронный реранкинг эффективным и масштабируемым для **экстремально больших списков кандидатов** (тысячи документов), сохраняя при этом высокую точность. Это достигается за счет комбинации архитектурных оптимизаций, позволяющих эффективно выполнять глубокое взаимодействие между запросом и документом, и новой стратегии обучения, учитывающей относительное ранжирование документов.

### Постановка задачи
Даны запрос $Q$ и отсортированный список из $N$ документов $D = \{d_1, d_2, ..., d_N\}$, полученный на первом этапе поиска. Задача реранкинга состоит в том, чтобы переупорядочить эти $N$ документов по степени их релевантности запросу $Q$, выдавая новый, более точный список.

### Альтернативные методы
На момент появления ExaRanker существовали следующие основные подходы к реранкингу:
*   **Разреженные реранкеры (Sparse Rerankers):** Такие как BM25. Быстрые, но основаны на совпадении ключевых слов и не учитывают семантическую релевантность. Часто используются на первом этапе поиска.
*   **Плотные реранкеры на основе Bi-Encoder архитектуры:** Например, ColBERT (2020). Используют две независимые башни (один энкодер для запроса, один для документа) для создания эмбеддингов, а затем измеряют их сходство (например, с помощью dot-product). Быстрые для инференса, так как эмбеддинги документов можно предварительно посчитать. Однако они менее эффективны для тонкого ранжирования, поскольку не позволяют глубокого взаимодействия между запросом и документом на уровне токенов.
*   **Кросс-энкодеры (Cross-Encoders):** Примеры включают MonoBERT (2019), MonoT5 (2021). Эти модели конкатенируют запрос и документ (`[CLS] Query [SEP] Document [SEP]`) и пропускают их через один общий Transformer энкодер. Такой подход позволяет глубокое взаимодействие на уровне токенов и демонстрирует высокую точность. Однако их главный недостаток — это высокая вычислительная стоимость, так как для каждой пары (запрос, документ) требуется полный проход через модель, что делает их неприменимыми для реранкинга тысяч документов.

ExaRanker отличается от них тем, что он стремится достичь точности кросс-энкодеров, но с эффективностью, позволяющей обрабатывать значительно большие списки документов, чем это было возможно ранее для моделей с глубоким взаимодействием.

### Архитектура
ExaRanker построен на основе мощного предварительно обученного **Transformer-энкодера (shared Transformer backbone)**, например, T5. Ключевые архитектурные особенности:
1.  **Общий Transformer-бэкбон (Shared Transformer Backbone):** Для каждой пары (запрос $Q$, документ $D_i$) вход подается в виде `[CLS] Q [SEP] D_i [SEP]` в один и тот же Transformer-энкодер. Использование *общего бэкбона* для всех пар означает, что основные параметры модели используются повторно, а эффективная обработка (например, батчирование, частичное кэширование) может снизить накладные расходы по сравнению с независимым вызовом модели для каждой пары.
2.  **Легковесный модуль парного взаимодействия (Lightweight Pairwise Module):** После обработки в общем бэкбоне, выходной эмбеддинг (например, [CLS]-токен) передается в относительно **легковесный модуль парного взаимодействия**. Этот модуль (может состоять из нескольких слоев Transformer или MLP) выполняет финальную, более глубокую и специфичную для ранжирования обработку, чтобы вычислить окончательную оценку релевантности. Он позволяет сохранить тонкое взаимодействие, характерное для кросс-энкодеров, но с меньшими вычислительными затратами, чем пропуск всего входа через полный, глубокий Transformer для каждой пары.
3.  **Выход:** Модель выдает скалярную оценку релевантности для каждой пары (запрос, документ).

### Алгоритм обучения
ExaRanker использует **стратегию обучения, учитывающую ранг (rank-aware training strategy)**, которая оптимизирует модель для эффективного ранжирования больших списков документов:
1.  **Формирование обучающих данных:** Для каждого запроса $Q$ формируется список документов, состоящий из одного или нескольких позитивных примеров (релевантных документов) и множества негативных примеров (нерелевантных документов). Особое внимание уделяется **майнингу сложных негативных примеров (hard negative mining)**, чтобы модель училась различать очень похожие, но нерелевантные документы.
2.  **Loss-функция, учитывающая ранг (Rank-aware Loss Function):** Вместо стандартных pointwise или pairwise loss-функций, ExaRanker использует специализированную loss-функцию, которая напрямую оптимизирует качество ранжирования. В статье упоминается "dedicated loss function", которая в сочетании с механизмом ренормализации способствует лучшему захвату относительного ранга документов в списке.
3.  **Групповая ренормализация (Group-wise Re-Normalization):** Это ключевая техника, используемая во время обучения. Оценки релевантности для всех документов в одном батче (или группе) **ренормализуются**, чтобы они были сопоставимы и отражали относительные ранги внутри группы. Этот механизм помогает модели лучше понимать относительную важность документов в контексте других кандидатов и стабилизирует обучение, особенно при работе с большим количеством негативных примеров.
4.  **Fine-tuning:** Модель инициализируется весами предварительно обученного Transformer (например, T5) и дообучается на датасетах для реранкинга (например, MS MARCO).

### Алгоритм инференса
1.  **Прием кандидатов:** Получает запрос $Q$ и список из $N$ предварительно извлеченных документов $D = \{d_1, d_2, ..., d_N\}$ от первого этапа поиска.
2.  **Парная обработка:** Для каждого документа $d_i$ из списка формируется вход `[CLS] Q [SEP] d_i [SEP]`. Эти входы могут быть объединены в батчи для параллельной обработки.
3.  **Вычисление оценок релевантности:** Каждый батч проходит через **общий Transformer-бэкбон**, а затем через **легковесный модуль парного взаимодействия**. В результате для каждой пары $(Q, d_i)$ получается скалярная оценка релевантности.
4.  **Сортировка:** Документы $d_i$ сортируются по убыванию их оценок релевантности.
5.  **Вывод:** Возвращается отсортированный список документов.

Благодаря оптимизациям в архитектуре и возможности эффективной обработки больших батчей, ExaRanker демонстрирует значительно меньшую задержку при инференсе для больших `N` по сравнению с обычными кросс-энкодерами, при этом сохраняя их высокую точность.

### Результаты
ExaRanker сравнивался с передовыми моделями на крупномасштабных бенчмарках, таких как MS MARCO и BEIR.
*   **MS MARCO:** Модель показала значительное улучшение метрики NDCG@10, превзойдя существующие state-of-the-art реранкеры (например, MonoT5) на 1-2 процентных пункта.
*   **Масштабируемость:** Наряду с улучшением точности, ExaRanker продемонстрировал существенно более быстрое время инференса, позволяя эффективно обрабатывать списки кандидатов, в несколько раз превышающие те, с которыми могли работать предыдущие кросс-энкодеры (например, реранкинг 1000 документов вместо 100). Это открывает возможности для использования более глубокого реранкинга в приложениях, где ранее это было невозможно из-за вычислительных ограничений.

## 📝 Критический анализ

```markdown
# ExaRanker (2023)
---
[[paper]](https://arxiv.org/pdf/2305.15609)

ExaRanker = eXtreme-scale Ranker, предложен Google Research в 2023 году.

ExaRanker — это модель для **нейронного реранкинга**, оптимизированная для обработки больших списков документов в информационном поиске, преодолевая ограничения традиционных кросс-энкодеров.

### Контекст
Современные системы поиска используют многоступенчатый подход: быстрый ретривер извлекает документы, а реранкер уточняет их порядок. Традиционные реранкеры на основе кросс-энкодеров медленны и дорогие для больших списков документов.

### Идея
ExaRanker делает реранкинг эффективным для **больших списков кандидатов** (тысячи документов), сохраняя точность. Это достигается архитектурными оптимизациями и новой стратегией обучения.

### Постановка задачи
Дан запрос $Q$ и список $N$ документов $D = \{d_1, d_2, ..., d_N\}$. Задача — переупорядочить документы по релевантности к $Q$.

### Альтернативные методы
* **Sparse Rerankers:** Быстрые, но не учитывают семантику.
* **Bi-Encoder:** Быстрые, но не обеспечивают глубокое взаимодействие.
* **Cross-Encoders:** Высокая точность, но дорогие для больших списков.

ExaRanker стремится к точности кросс-энкодеров с эффективностью для больших списков.

### Архитектура
ExaRanker использует **Transformer-энкодер** (например, T5):
1. **Shared Transformer Backbone:** Вход `[CLS] Q [SEP] D_i [SEP]` обрабатывается общим энкодером.
2. **Lightweight Pairwise Module:** Обрабатывает выходной эмбеддинг для окончательной оценки релевантности.
3. **Выход:** Скалярная оценка релевантности для каждой пары.

### Алгоритм обучения
1. **Формирование данных:** Списки с позитивными и негативными примерами.
2. **Rank-aware Loss Function:** Оптимизирует качество ранжирования.
3. **Group-wise Re-Normalization:** Ренормализует оценки в батче.
4. **Fine-tuning:** Дообучение на реранкинг датасетах.

### Алгоритм инференса
1. **Прием кандидатов:** Запрос $Q$ и список $N$ документов.
2. **Парная обработка:** Входы `[CLS] Q [SEP] d_i [SEP]` обрабатываются батчами.
3. **Вычисление оценок:** Через **Transformer-бэкбон** и **Pairwise Module**.
4. **Сортировка:** По убыванию оценок релевантности.
5. **Вывод:** Отсортированный список документов.

<img src="img/img.png" width=500>

### Результаты
ExaRanker улучшил NDCG@10 на MS MARCO на 1-2 п.п. и показал более быстрое время инференса, позволяя реранкинг 1000 документов вместо 100.
```


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Инициализация модели и токенизатора
tokenizer = T5Tokenizer.from_pretrained('t5-small')
model = T5ForConditionalGeneration.from_pretrained('t5-small')

# Пример запроса и документов
query = "What is the capital of France?"
documents = [
    "Paris is the capital of France.",
    "Berlin is the capital of Germany.",
    "Madrid is the capital of Spain."
]

# Функция для подготовки входных данных в формате [CLS] Q [SEP] D_i [SEP]
def prepare_input(query, document):
    return f"rank: {query} </s> {document}"

# Подготовка входных данных
inputs = [prepare_input(query, doc) for doc in documents]

# Токенизация и создание тензоров
input_encodings = tokenizer(inputs, padding=True, truncation=True, return_tensors="pt")

# Пропуск через общий Transformer-бэкбон
outputs = model(input_ids=input_encodings['input_ids'], attention_mask=input_encodings['attention_mask'])

# Извлечение эмбеддингов [CLS]-токена
# В T5, [CLS] токен не используется, но мы можем взять первый токен как представление
cls_embeddings = outputs.last_hidden_state[:, 0, :]

# Легковесный модуль парного взаимодействия (например, простой MLP)
class PairwiseInteractionModule(torch.nn.Module):
    def __init__(self, input_dim):
        super(PairwiseInteractionModule, self).__init__()
        self.linear = torch.nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x)

# Инициализация легковесного модуля
pairwise_module = PairwiseInteractionModule(cls_embeddings.size(-1))

# Вычисление оценок релевантности
relevance_scores = pairwise_module(cls_embeddings).squeeze()

# Сортировка документов по оценкам релевантности
sorted_indices = torch.argsort(relevance_scores, descending=True)
sorted_documents = [documents[i] for i in sorted_indices]

# Вывод отсортированных документов
print("Sorted Documents by Relevance:")
for doc in sorted_documents:
    print(doc)

# Пример вывода:
# Sorted Documents by Relevance:
# Paris is the capital of France.
# Madrid is the capital of Spain.
# Berlin is the capital of Germany.
```

### Ключевые моменты:
1. **Общий Transformer-бэкбон:** Входные данные формируются в формате `[CLS] Q [SEP] D_i [SEP]` и пропускаются через один и тот же Transformer-энкодер (T5 в данном примере).

2. **Легковесный модуль парного взаимодействия:** После получения эмбеддингов из Transformer, они передаются в легковесный модуль (в данном случае, простой линейный слой), который вычисляет окончательные оценки релевантности.

3. **Эффективная обработка:** Входные данные обрабатываются батчами, что позволяет эффективно использовать вычислительные ресурсы и снижает накладные расходы.

4. **Сортировка и вывод:** Документы сортируются по убыванию их оценок релевантности, что позволяет получить более точный список документов.

Этот код иллюстрирует основные архитектурные особенности ExaRanker, такие как использование общего Transformer-бэкбона и легковесного модуля парного взаимодействия, что позволяет эффективно обрабатывать большие списки документов.